In [ ]:
####################################
#ENVIRONMENT SETUP

In [ ]:
#Importing Libraries
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import ScalarFormatter
import matplotlib.gridspec as gridspec
import xarray as xr

import sys; import os; import time; from datetime import timedelta
import pickle
import h5py
from tqdm import tqdm

In [ ]:
#MAIN DIRECTORIES
def GetDirectories():
    mainDirectory='/mnt/lustre/koa/koastore/torri_group/air_directory/Projects/DCI-Project/'
    mainCodeDirectory=os.path.join(mainDirectory,"Code/CodeFiles/")
    scratchDirectory='/mnt/lustre/koa/scratch/air673/'
    codeDirectory=os.getcwd()
    return mainDirectory,mainCodeDirectory,scratchDirectory,codeDirectory

[mainDirectory,mainCodeDirectory,scratchDirectory,codeDirectory] = GetDirectories()

In [ ]:
#IMPORT CLASSES (from current directory)
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
from CLASSES_Variable_Calculation import ModelData_Class, SlurmJobArray_Class, DataManager_Class

In [ ]:
#IMPORT FUNCTIONS
sys.path.append(os.path.join(mainCodeDirectory,"Variable_Calculation"))
import FUNCTIONS_Variable_Calculation
from FUNCTIONS_Variable_Calculation import * # import NumericalFunctions 

#IMPORT FUNCTIONS
import sys
path=os.path.join(mainCodeDirectory,'Functions/')
sys.path.append(path)

import NumericalFunctions
from NumericalFunctions import * # import NumericalFunctions 
import PlottingFunctions
from PlottingFunctions import * # import PlottingFunctions

# # Get all functions in NumericalFunctions
# import inspect
# functions = [f[0] for f in inspect.getmembers(NumericalFunctions, inspect.isfunction)]
# functions

In [ ]:
####################################
#LOADING CLASSES

In [ ]:
#data loading class
ModelData = ModelData_Class(mainDirectory, scratchDirectory, simulationNumber=1)
#data manager class
DataManager = DataManager_Class(mainDirectory, scratchDirectory, ModelData.res, ModelData.t_res, ModelData.Nz_str,
                                ModelData.Np_str, dataType="CalculateMoreVariables", dataName="VariablePerturbations",
                                dtype='float32')

In [ ]:
#JOB ARRAY SETUP
UsingJobArray=True

def GetNumJobs(res,t_res):
    if res=='1km':
        if t_res=='5min':
            num_jobs=20
        elif t_res=='1min':
            num_jobs=100
    elif res=='250m': 
        if t_res=='1min':
            num_jobs=500
    return num_jobs
num_jobs = GetNumJobs(ModelData.res,ModelData.t_res)
SlurmJobArray = SlurmJobArray_Class(total_elements=ModelData.Ntime, num_jobs=num_jobs, UsingJobArray=UsingJobArray)
start_job = SlurmJobArray.start_job; end_job = SlurmJobArray.end_job

def GetNumElements():
    loop_elements = np.arange(ModelData.Ntime)[start_job:end_job]
    return loop_elements
loop_elements = GetNumElements()

In [ ]:
####################################
#DATA LOADING FUNCTIONS

In [ ]:
def GetVariableNames():
    variableNames = ['qv','RH_vapor']
    variableNames += ['theta_v','theta_e','MSE']
    variableNames += ['winterp','VMF_g','VMF_c','qcqi','HMC']
    return variableNames
def GetInputVariables(DataManager, timeString):
    variableNames = GetVariableNames()
    
    # inputDataDictionary = {variableName: CallVariable(ModelData, DataManager, timeString, variableName)\
    #                        for variableName in tqdm(variableNames, desc="Loading Data",leave=False)}
    inputDataDictionary = {variableName: CallVariable(ModelData, DataManager, timeString, variableName)\
                           for variableName in variableNames}
    return inputDataDictionary

In [ ]:
#Getting SBF locations for each time

#IMPORT CLASSES
sys.path.append(os.path.join(mainCodeDirectory,"3_Project_Algorithms","2_Tracking_Algorithms"))
from CLASSES_TrackingAlgorithms import TrackingAlgorithms_DataLoading_Class, SlurmJobArray_Class, Results_InputOutput_Class, TrackedParcel_Loading_Class


def Get_AvgConvergence(t):

    timeString = ModelData.timeStrings[t]
    outputDataDirectory=os.path.normpath(os.path.join(DataManager.outputDataDirectory,"..","Eulerian_CLTracking"))
    Dictionary = TrackingAlgorithms_DataLoading_Class.LoadData(ModelData, DataManager, timeString,
                     dataName="Eulerian_CLTracking",outputDataDirectory=outputDataDirectory,printstatement=False)
    avgConvergence = Dictionary["avgConvergence"]
    return avgConvergence

def find_SBF_xmaxs():
    xmaxs=[]
    for t in tqdm(range(ModelData.Ntime)):
        if t == 0:
            xmaxs.append(np.nan)
        else:
            avgConvergence = Get_AvgConvergence(t)
            avgConvergence_max=np.nanmax(avgConvergence)
            xmax = np.where(avgConvergence==avgConvergence_max)[0][0]
            xmaxs.append(xmax)
    return xmaxs

def Get_SBF_xmaxs_FilePath(ModelData, DataManager):
    fileName = (
        f"SBF_xmaxs_{ModelData.res}_{ModelData.t_res}_"
        f"{ModelData.Ntime}nt.pkl"
    )
    return os.path.join(DataManager.outputDataDirectory, fileName)

def LoadOrRun_find_SBF_xmaxs(ModelData, DataManager, overwrite=False):
    filePath = Get_SBF_xmaxs_FilePath(ModelData, DataManager)
    
    if os.path.exists(filePath) and not overwrite:
        print(f"reading from {filePath}")
        with open(filePath, "rb") as f:
            xmaxs = pickle.load(f)
        return xmaxs

    xmaxs = find_SBF_xmaxs()

    os.makedirs(os.path.dirname(filePath), exist_ok=True)
    with open(filePath, "wb") as f:
        pickle.dump(xmaxs, f)
    print(f"saved to {filePath}")
    return xmaxs
DataManager_Eulerian_CLTracking = DataManager_Class(mainDirectory, scratchDirectory, ModelData.res, ModelData.t_res, ModelData.Nz_str,
                                ModelData.Np_str, dataType="Tracking_Algorithms", dataName="Eulerian_CLTracking",
                                dtype='float32',codeSection = "Project_Algorithms")
xmaxs = LoadOrRun_find_SBF_xmaxs(ModelData, DataManager_Eulerian_CLTracking)
coastlineX = int(ModelData.Nxh*0.25)
xmaxs[0]=xmaxs[1] #correct nan at time 0

In [ ]:
####################################
#CALCULATION FUNCTIONS

In [ ]:
#Calculation Functions
def CalculateVariablePerturbation(variable,meanType='SBF_relative',t=None):
    
    if meanType == 'domain_wide':
        variableMean = np.mean(variable,axis=(1,2))
        variablePerturbation = variable - variableMean[:,np.newaxis,np.newaxis]
    # elif meanType == 'right_of_coast': #depreciated for "coast_relative"
    #     variableMean = np.mean(variable[:,:,coastlineX:],axis=(1,2))
    #     variablePerturbation = variable - variableMean[:,np.newaxis,np.newaxis]
    # elif meanType == 'right_of_SBF': #depreciated for "SBF_relative"
    #     variableMean = np.mean(variable[:,:,xmaxs[t]:],axis=(1,2))
    #     variablePerturbation = variable - variableMean[:,np.newaxis,np.newaxis]
    elif meanType == 'coast_relative':
        variableMean_left = np.mean(variable[:,:,:coastlineX+1],axis=(1,2))
        variableMean_right = np.mean(variable[:,:,coastlineX:],axis=(1,2))
        variablePerturbation = variable.copy()
        variablePerturbation[:,:,:coastlineX+1] -= variableMean_left[:,np.newaxis,np.newaxis]
        variablePerturbation[:,:,coastlineX:] -= variableMean_right[:,np.newaxis,np.newaxis]
        variableMean=(variableMean_right-variableMean_left) #placeholder
    elif meanType == 'SBF_relative':
        variableMean_left = np.mean(variable[:,:,:xmaxs[t]+1],axis=(1,2))
        variableMean_right = np.mean(variable[:,:,xmaxs[t]:],axis=(1,2))
        variablePerturbation = variable.copy()
        variablePerturbation[:,:,:xmaxs[t]+1] -= variableMean_left[:,np.newaxis,np.newaxis]
        variablePerturbation[:,:,xmaxs[t]:] -= variableMean_right[:,np.newaxis,np.newaxis]
        variableMean=(variableMean_right-variableMean_left) #placeholder
        
    return variablePerturbation,variableMean

def CalculateVariablePerturbationDictionary(inputDataDictionary,t=None):

    variableNames = GetVariableNames()
    # outputDataDictionary = {f"{variableName}_prime": CalculateVariablePerturbation(variable=inputDataDictionary[variableName],t=t)[0]\
    #                        for variableName in tqdm(variableNames, desc="Loading Data",leave=False)}
    outputDataDictionary = {f"{variableName}_prime": CalculateVariablePerturbation(variable=inputDataDictionary[variableName],t=t)[0]\
                           for variableName in variableNames}

    return outputDataDictionary

In [ ]:
####################################
#RUNNING

In [ ]:
#CALCULATING AND APPENDING TO DATA EACH TIMESTEP
for t in tqdm(loop_elements, total=len(loop_elements)):
    if np.mod(t,1)==0: print(f'Current time {t}')

    #getting timestring for loading input data
    timeString = ModelData.timeStrings[t]

    #loading input variables
    inputDataDictionary = GetInputVariables(DataManager, timeString)

    #calculating perturbations
    outputDataDictionary = CalculateVariablePerturbationDictionary(inputDataDictionary,t=t)
    
    #outputting
    DataManager.SaveOutputTimestep(DataManager.outputDataDirectory, timeString, outputDataDictionary)

In [ ]:
######################################################

In [ ]:
# #READING BACK IN
# def GetVariableData(t, dataName="Virtual_Potential_Temperature"):
#     res = ModelData.res
#     t_res = ModelData.t_res
#     Nz_str = ModelData.Nz_str
#     inputDirectory = os.path.join(DataManager.outputDirectory, f"{res}_{t_res}_{Nz_str}nz", dataName)
#     timeString = ModelData.timeStrings[t]

#     FileName = os.path.join(inputDirectory, f"{dataName}_{res}_{t_res}_{Nz_str}nz_{timeString}.h5")

#     dataDictionary = {}
#     with h5py.File(FileName, 'r') as f:
#         print("Keys in file:", list(f.keys()))
#         for key in f.keys():
#             dataDictionary[key] = f[key][:]
#             print(f"{key}: shape = {dataDictionary[key].shape}, dtype = {dataDictionary[key].dtype}")
#     return dataDictionary

# dataDictionary = GetVariableData(t=0)

In [ ]:
####################################
#TESTING

In [ ]:
# #getting input data for single timestep
# t=100
# timeString = ModelData.timeStrings[t]
# inputDataDictionary = GetInputVariables(DataManager, timeString)

# def GetLevels_mean(data, n=20):
#     return np.linspace(-np.max(data), np.max(data), 20)

# def GetLevels_percentile(data, p=99, n=20, eps=1e-12):
#     dataAbs = np.abs(data)
#     dataAbs = dataAbs[np.isfinite(dataAbs)]
#     dataAbs = dataAbs[dataAbs > eps]   # ignore near-zero values

#     if len(dataAbs) == 0:
#         vmax = eps
#     else:
#         vmax = np.percentile(dataAbs, p)

#     return np.linspace(-vmax, vmax, n)

# #COAST RELATIVE
# #calculating
# varName='qv'
# # [domainWide,_] = CalculateVariablePerturbation(variable=inputDataDictionary[varName], meanType=meanType, t=t)

# meanType='domain_wide'; center_XLocation = coastlineX
# [variablePerturbation_domainWide,_] = CalculateVariablePerturbation(variable=inputDataDictionary[varName], meanType=meanType, t=t)

# meanType='coast_relative'; center_XLocation = coastlineX
# meanType='SBF_relative'; center_XLocation = xmaxs[t]
# [variablePerturbation,_] = CalculateVariablePerturbation(variable=inputDataDictionary[varName], meanType=meanType, t=t)

# domain = (variablePerturbation_domainWide)[:,100]
# left=(variablePerturbation[:,:,:center_XLocation+1])[:,100]
# right=(variablePerturbation[:,:,center_XLocation:])[:,100]
# domain_mean = np.mean(domain, axis=1); left_mean=np.mean(left,axis=1);right_mean=np.mean(right,axis=1)

# # levels_domain = GetLevels_mean(domain)
# # levels_left   = GetLevels_mean(left)
# # levels_right  = GetLevels_mean(right)
# levels_domain = GetLevels_percentile(domain)
# levels_left   = GetLevels_percentile(left)
# levels_right  = GetLevels_percentile(right)

# #plotting
# xh = ModelData.xh - ModelData.xh[0]
# fig, axes = plt.subplots(2, 3, figsize=(16,8), sharey=True)
# ax1, ax2, ax3, ax4, ax5, ax6 = axes.flatten()
# cmap = 'RdBu_r'
# xh = ModelData.xh - ModelData.xh[0]

# # --- DOMAIN-WIDE ---
# cf0 = ax1.contourf(xh, ModelData.zh, domain, levels=levels_domain, cmap=cmap, extend='both')
# cbar0 = fig.colorbar(cf0, ax=ax1)
# ax1.set_title("Domain-wide")
# ax1.set_xlabel('x (km)')
# ax1.set_ylabel('z (km)')
# ax1.axvline(center_XLocation,color='black',linestyle='--',zorder=50)

# ax4.plot(domain_mean, ModelData.zh)
# ax4.set_xlabel(f"{varName}_prime")

# # --- LEFT ---
# cf1 = ax2.contourf(xh[:center_XLocation+1], ModelData.zh, left,
#                    levels=levels_left, cmap=cmap, extend='both')
# cbar1 = fig.colorbar(cf1, ax=ax2)
# ax2.set_title(f"Left of {meanType.split('_')[0]}")
# ax2.set_xlabel('x (km)')

# ax5.plot(left_mean, ModelData.zh)
# ax5.set_xlabel(f"{varName}_prime")

# # --- RIGHT ---
# cf2 = ax3.contourf(xh[center_XLocation:], ModelData.zh, right,
#                    levels=levels_right, cmap=cmap, extend='both')
# cbar2 = fig.colorbar(cf2, ax=ax3)
# ax3.set_title(f"Right of {meanType.split('_')[0]}")
# ax3.set_xlabel('x (km)')

# ax6.plot(right_mean, ModelData.zh)
# ax6.set_xlabel(f"{varName}_prime")

# # --- formatting ---
# apply_scientific_notation_colorbar([cbar0, cbar1, cbar2])
# apply_scientific_notation([ax4, ax5, ax6])

# for ax in [ax1, ax2, ax3, ax4, ax5, ax6]:
#     ax.set_ylim(bottom=0)

# #comparing axis limits
# axes_list = [ax4, ax5, ax6]
# xmin = min(ax.get_xlim()[0] for ax in axes_list)
# xmax = max(ax.get_xlim()[1] for ax in axes_list)
# for ax in axes_list:
#     ax.set_xlim(xmin, xmax)


# plt.suptitle(f"Comparing Various Perturbation Calculations Relative to {meanType.split('_')[0]} at t = {t*ModelData.dt/3600+6:.1f} hrs")
# plt.tight_layout()